In [4]:
import os
import tkinter as tk
from tkinter import filedialog

# Inicializa o Tk sem abrir janela extra
root = tk.Tk()
root.withdraw()

# Abre o explorador de arquivos
FILE_PATH = filedialog.askopenfilename(
    title="Selecione o documento",
    filetypes=[
        ("Documentos", "*.pdf *.docx *.txt"),
        ("PDF", "*.pdf"),
        ("Todos os arquivos", "*.*")
    ]
)

if not FILE_PATH:
    raise RuntimeError("Nenhum arquivo selecionado")

FILE_PATH = os.path.abspath(FILE_PATH)

print("Arquivo selecionado:", os.path.basename(FILE_PATH))
print("Caminho completo:", FILE_PATH)


Arquivo selecionado: Documento sobre RAG-ENG.pdf
Caminho completo: /home/brain/projects/RAG para PDFs/RAG com OCR do Vini/ROA - RAG com OCR Academico/Documentos/Documento sobre RAG-ENG.pdf


In [ ]:
type(result)
dir(result)


In [11]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert(FILE_PATH)

doc = result.document  # ← AQUI está o documento real


[INFO] 2026-02-04 14:48:19,469 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-04 14:48:19,470 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-02-04 14:48:19,481 [RapidOCR] download_file.py:60: File exists and is valid: /home/brain/miniconda3/envs/ROAv2/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-04 14:48:19,482 [RapidOCR] main.py:50: Using /home/brain/miniconda3/envs/ROAv2/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-04 14:48:19,648 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-04 14:48:19,648 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-02-04 14:48:19,650 [RapidOCR] download_file.py:60: File exists and is valid: /home/brain/miniconda3/envs/ROAv2/lib/python3.13/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-02-04 14:48:19,650 [RapidOCR] main.py:50: Using /home/brain/miniconda3/envs/ROAv2/lib/

In [12]:
text = doc.export_to_text()
print(text[:1000])


Parameter `strict_text` has been deprecated and will be ignored.


## Developing Retrieval Augmented Generation (RAG) based LLM Systems from PDFs: An Experience Report

Ayman Asad Khan Tampere University ayman.khan@tuni.fi

Kai Kristian Kemell Tampere University kai-kristian.kemell@tuni.fi

Md Toufique Hasan Tampere University mdtoufique.hasan@tuni.fi

Jussi Rasku Tampere University jussi.rasku@tuni.fi

Pekka Abrahamsson Tampere University pekka.abrahamsson@tuni.fi

Abstract. This paper presents an experience report on the development of Retrieval Augmented Generation (RAG) systems using PDF documents as the primary data source. The RAG architecture combines generative capabilities of Large Language Models (LLMs) with the precision of information retrieval. This approach has the potential to redefine how we interact with and augment both structured and unstructured knowledge in generative models to enhance transparency, accuracy and contextuality of responses. The paper details the end-to-end pipeline, from data collection, preprocessing, to retrieval

In [13]:
print(type(result))
print(dir(result))

print("\n--- DOCUMENT ---\n")
print(type(doc))
print(dir(doc))


<class 'docling.datamodel.document.ConversionResult'>
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__firstlineno__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic_root_model__',

In [16]:
native_metadata = {
    "name": doc.name,
    "origin": str(doc.origin),
    "num_pages": doc.num_pages(),
    "version": doc.version,
    "has_tables": len(doc.tables) > 0,
    "has_images": len(doc.pictures) > 0,
    "num_text_blocks": len(doc.texts),
}

for k, v in native_metadata.items():
    print(f"{k}: {v}")


name: Documento sobre RAG-ENG
origin: mimetype='application/pdf' binary_hash=8162572176947223712 filename='Documento sobre RAG-ENG.pdf' uri=None
num_pages: 36
version: 1.9.0
has_tables: True
has_images: True
num_text_blocks: 483


In [17]:
native_metadata = {
    "name": doc.name,
    "origin_filename": doc.origin.filename,
    "mimetype": doc.origin.mimetype,
    "num_pages": doc.num_pages(),        # ← CHAMADA CORRETA
    "docling_version": doc.version,
    "num_text_blocks": len(doc.texts),
    "num_tables": len(doc.tables),
    "num_images": len(doc.pictures),
}

for k, v in native_metadata.items():
    print(f"{k}: {v}")


name: Documento sobre RAG-ENG
origin_filename: Documento sobre RAG-ENG.pdf
mimetype: application/pdf
num_pages: 36
docling_version: 1.9.0
num_text_blocks: 483
num_tables: 4
num_images: 7


In [15]:
roa_metadata = {
    "source_type": "livro",          # manual por enquanto
    "authority_level": "alta",       # manual
    "domain": "veterinaria",         # manual
    "language": "pt-BR",             # depois automatiza
    "ingestion_method": "manual",
    "doc_name": doc.name,
    "num_pages": doc.num_pages,
}
